#Add Dependencies

In [27]:
import openai
from qdrant_client import QdrantClient
from langsmith import traceable, get_current_run_tree

In [28]:
from dotenv import load_dotenv
load_dotenv("../../.env")

True

In [29]:
import os

print(os.getenv("LANGSMITH_TRACING"))
print(bool(os.getenv("LANGSMITH_API_KEY")))
print(os.getenv("LANGSMITH_PROJECT"))

true
True
None


In [30]:
@traceable(
    run_type="embedding",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "text-embedding-3-small",
    },
)
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    run_tree = get_current_run_tree()
    if run_tree is not None and response.usage is not None:
        run_tree.set(
            usage_metadata={
                "input_tokens": response.usage.prompt_tokens,
                "output_tokens": 0,
                "total_tokens": response.usage.total_tokens,
            }
        )

    return response.data[0].embedding

In [31]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [32]:
@traceable(run_type="retriever")
def retrieve_data(query, collection_name="toys_and_games_small_sample", k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=k
    )
    retrieved_context_id = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings=[]

    for result in search_result.points:
        retrieved_context_id.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["all_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_id": retrieved_context_id,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

### Format retrieved context function

In [33]:
@traceable(run_type="chain")
def process_context(retrieved_context):
    formatted_context = []

    for index, (product_id, description, rating, score) in enumerate(
        zip(
            retrieved_context["retrieved_context_id"],
            retrieved_context["retrieved_context"],
            retrieved_context["retrieved_context_ratings"],
            retrieved_context["similarity_scores"],
            strict=True,
        ),
        start=1,
    ):
        formatted_context.append(
            f"Context {index}\n"
            f"Product ID: {product_id}\n"
            f"Rating: {rating}/5\n"
            f"Similarity: {score:.3f}\n"
            f"Description: {description}"
        )

    return "\n\n---\n\n".join(formatted_context)

### Create Prompt Template Function

In [34]:
@traceable(run_type="prompt")
def build_prompt(preprocessed_context, query):
    prompt = f"""
You are a shopping assistant that can answer question about the products in stock.
You will be given a question a nd a list of context.
Instruction:
 - Answer the question based on the context provided onlym
 - Never use word context and refer to it as the available products
 - Do not use markdown formatting.
 
 Context:
{preprocessed_context}
Question:
{query}

 """
    return prompt

In [35]:
@traceable(
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-6-luna",
    },
)
def generate_asnwer(prompt):
    response = openai.chat.completions.create(
        model="gpt-6-luna",
        messages=[
            {"role": "system", "content": prompt}
        ],
    )

    run_tree = get_current_run_tree()
    if run_tree is not None and response.usage is not None:
        run_tree.set(
            usage_metadata={
                "input_tokens": response.usage.prompt_tokens,
                "output_tokens": response.usage.completion_tokens,
                "total_tokens": response.usage.total_tokens,
            }
        )

    return response.choices[0].message.content

### Combined RAG pipline

In [36]:
@traceable(run_type="chain")
def rag_pipline(query, k=5):
    retrieved_context = retrieve_data(query,collection_name="toys_and_games_small_sample", k=k)
    preprocessed_context=process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_asnwer(prompt)

    return answer

In [37]:
print(rag_pipline("Recommend some toy that can be recharged via USB"))

The available products with rechargeable batteries and USB charging are:

• Kidstinct RC Tank (B0BC8TJGQP): Includes two rechargeable batteries and a USB charger, with up to 60 minutes of playtime.
• LANSAND RC Excavator (B09XWB3M5X): Includes two rechargeable batteries and a USB charger, with up to 60 minutes of playtime.


In [38]:
print(rag_pipline("Recommend most popular toy for 5 year old boy"))

The Hot Wheels Monster Truck 1:24 Scale Bone Shaker is a great pick for a 5-year-old boy. It’s rated 4.8/5, designed for ages 3–8, and has giant wheels for crashing and stunt play.


In [39]:
from langsmith import Client, traceable

client = Client()
print([project.name for project in client.list_projects(limit=20)])

@traceable(name="notebook-smoke-test", run_type="chain")
def smoke_test():
    return "ok"

smoke_test()

['rag-tracing-production', 'rag-tracing-staging', 'rag-tracing-development', 'rag-tracing-local']


'ok'